In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahdiislam/pressure-sensor-heatmaprgb")

print("Path to dataset files:", path)

Path to dataset files: /Users/ddevatha/.cache/kagglehub/datasets/mahdiislam/pressure-sensor-heatmaprgb/versions/1


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [ ]:
transforms = T.Compose([
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [4]:
import pickle
import numpy as np

with open(f'{path}/X_9_RGB.pickle', 'rb') as f:
    X = pickle.load(f)

with open(f'{path}/y_9_RGB.pickle', 'rb') as f:
    y = pickle.load(f)

In [5]:
from torch.utils.data import Dataset

class HeatmapDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx]  # should be C × H × W for torchvision transforms
        label = self.y[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

In [6]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.int64)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67, shuffle=True)

X_train = X_train.permute(0, 3, 1, 2)
X_test = X_test.permute(0, 3, 1, 2)

train_data = HeatmapDataset(X_train, y_train, transform=transforms)
test_data = HeatmapDataset(X_test, y_test, transform=transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
class HeatCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3) # (3, 224, 224) -> (32, 222, 222)
        self.pool = nn.MaxPool2d(2, 2) # (32, 222, 222) -> (32, 111, 111)
        self.conv2 = nn.Conv2d(32, 64, 3) # (32, 111, 111) -> (64, 109, 109)
        # pooled: (64, 109, 109) -> (64, 54, 54)

        self.fc1 = nn.Linear(64 * 54 * 54, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 9)

    def forward(self, x):
        out = self.pool(F.relu(self.conv1(x)))
        out = self.pool(F.relu(self.conv2(out)))

        out = out.flatten(1)
        out = F.relu(self.fc1(out))
        out = F.relu(self.fc2(out))

        out = self.out(out)
        return out   

In [8]:
model = HeatCNN()
model.train()

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

for epoch in range(30):
    total_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch {epoch + 1}: Loss >>> {total_loss / len(train_loader):.4f}') 

Epoch 1: Loss >>> 5.0283
Epoch 2: Loss >>> 1.2376
Epoch 3: Loss >>> 0.4788
Epoch 4: Loss >>> 0.1883
Epoch 5: Loss >>> 0.0930
Epoch 6: Loss >>> 0.0976
Epoch 7: Loss >>> 0.0615
Epoch 8: Loss >>> 0.0622
Epoch 9: Loss >>> 0.0561
Epoch 10: Loss >>> 0.0524
Epoch 11: Loss >>> 0.0219
Epoch 12: Loss >>> 0.0291
Epoch 13: Loss >>> 0.0408
Epoch 14: Loss >>> 0.0343
Epoch 15: Loss >>> 0.0356
Epoch 16: Loss >>> 0.0458
Epoch 17: Loss >>> 0.0414
Epoch 18: Loss >>> 0.0335
Epoch 19: Loss >>> 0.0234
Epoch 20: Loss >>> 0.0503
Epoch 21: Loss >>> 0.0397
Epoch 22: Loss >>> 0.0297
Epoch 23: Loss >>> 0.0213
Epoch 24: Loss >>> 0.0170
Epoch 25: Loss >>> 0.0305
Epoch 26: Loss >>> 0.0268
Epoch 27: Loss >>> 0.0251
Epoch 28: Loss >>> 0.0257
Epoch 29: Loss >>> 0.0196
Epoch 30: Loss >>> 0.0173


In [10]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)

        _, predictions = torch.max(output, 1)

        all_preds.extend(predictions)
        all_labels.extend(labels)

print(classification_report(all_labels, all_preds, target_names=['Normal foot', 'Left foot forward leaned', 'Right foot forward leaned', 'Left foot backward leaned', 'Right foot backward leaned', 'Left sided lean', 'Right sided lean', 'Left foot twisted', 'Right foot twisted']))

                            precision    recall  f1-score   support

               Normal foot       1.00      1.00      1.00        37
  Left foot forward leaned       0.97      1.00      0.99        35
 Right foot forward leaned       1.00      0.97      0.98        32
 Left foot backward leaned       0.98      1.00      0.99        49
Right foot backward leaned       1.00      0.98      0.99        43
           Left sided lean       1.00      1.00      1.00        56
          Right sided lean       1.00      1.00      1.00        30
         Left foot twisted       1.00      1.00      1.00        37
        Right foot twisted       1.00      1.00      1.00        42

                  accuracy                           0.99       361
                 macro avg       0.99      0.99      0.99       361
              weighted avg       0.99      0.99      0.99       361



In [16]:
torch.save(model.state_dict(), "heatmap_model_state.pth")